In [1]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import pandas as pd
from tokenizers import Tokenizer
from functools import reduce

In [2]:
train_data = open("./train.txt").read()
valid_data = open("./val.txt").read()

In [3]:
tokenizer = Tokenizer.from_file("tokenizer.json")

In [4]:
device = torch.device('mps')

In [5]:
seq_len = 72          # Must be large to capture rhymes and meter
vocab_size = tokenizer.get_vocab_size()     # BPE target size
n_hidden = 256        # Brain power for 17th-century French
n_layers = 3          # Network depth
p = 0.3               # Dropout
batch_size = 64       # Increased to 64 for SGD stability!
alpha = 0.1           # AR 
beta = 0.05           # TAR
momentum = 0.9        # Standard momentum
lr = 1              # Starting Learning Rate
epochs = 15
max_lr = 5e-3
wd = 0.01

In [6]:
class LM(nn.Module):
    def __init__(self, vocab_size, n_hidden, n_layers, p_dropout):
        super().__init__()
        self.n_hidden = n_hidden
        self.n_layers = n_layers
        
        # 1. Native Embedding Layer
        self.embedding = nn.Embedding(vocab_size, n_hidden)
        
        # 2. Native LSTM Layer
        # batch_first=True tells PyTorch to expect inputs of shape (batch, seq, features)
        # dropout inside nn.LSTM applies "Zaremba" vertical dropout between the layers automatically!
        self.lstm = nn.LSTM(
            input_size=n_hidden, 
            hidden_size=n_hidden, 
            num_layers=n_layers, 
            dropout=p_dropout if n_layers > 1 else 0,
            batch_first=True 
        )
        
        # 3. Standard Dropout for embeddings and final output
        self.dropout = nn.Dropout(p_dropout)
        
        # 4. Native Linear Output Layer
        self.fc = nn.Linear(n_hidden, vocab_size)
        
        # 5. THE MAGIC TRICK: Weight Tying
        # By setting the output weights to be exactly the embedding weights, 
        # we cut the model's total parameters in half and massively reduce overfitting.
        self.fc.weight = self.embedding.weight
        
        self.hidden_state = None

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        bs = x.size(0)
        
        # Dynamically initialize or resize hidden states
        if self.hidden_state is None or self.hidden_state[0].size(1) != bs:
            weight = next(self.parameters()).data
            # PyTorch native LSTMs expect states shaped: (num_layers, batch_size, hidden_size)
            self.hidden_state = (weight.new_zeros(self.n_layers, bs, self.n_hidden),
                                 weight.new_zeros(self.n_layers, bs, self.n_hidden))
            
        # 1. Embedding + Dropout
        emb = self.dropout(self.embedding(x))
        
        # 2. C++ Optimized LSTM 
        # No more Python loops! PyTorch handles the entire sequence length instantly.
        out, self.hidden_state = self.lstm(emb, self.hidden_state)
        
        # 3. Truncated BPTT (Detach to prevent memory leaks)
        self.hidden_state = (self.hidden_state[0].detach(), self.hidden_state[1].detach())
        
        # 4. Output Dropout + Linear Projection
        out = self.dropout(out)
        logits = self.fc(out) 
        
        return logits

    def reset(self):
        self.hidden_state = None

In [7]:
loss_fn = nn.CrossEntropyLoss()

In [8]:
class FontaineDataset(torch.utils.data.IterableDataset):
    def __init__(self, data, seq_len, batch_size):
        super().__init__()
        text_tensor = torch.tensor(tokenizer.encode(data).ids, device=device)
        
        n_tokens = len(text_tensor) - 1 
        tokens_per_stream = n_tokens // batch_size
        
        x_data = text_tensor[:batch_size * tokens_per_stream]
        y_data = text_tensor[1 : batch_size * tokens_per_stream + 1]
        
        x_data = x_data.view(batch_size, -1)
        y_data = y_data.view(batch_size, -1)
        
        self.batches = []
        
        for i in range(0, x_data.shape[1] - seq_len + 1, seq_len):
            x_chunk = x_data[:, i:i+seq_len]
            y_chunk = y_data[:, i:i+seq_len]
            
            if x_chunk.shape[1] == seq_len:
                self.batches.append((x_chunk, y_chunk))

    def __iter__(self):
        return iter(self.batches)

In [9]:
ds       = FontaineDataset(train_data, seq_len, batch_size)
valid_ds = FontaineDataset(valid_data, seq_len, batch_size)

In [10]:
model = LM(vocab_size, n_hidden, n_layers, p).to(device)

In [11]:
optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=wd)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=max_lr, 
    steps_per_epoch=len(ds.batches), 
    epochs=epochs,
    pct_start=0.3
)

for e in range(epochs):
    model.train()
    total_loss, steps = 0.0, 0
    
    for x, y in ds:
        if x.shape == torch.Size([batch_size, seq_len]):
            x, y = x.to(device), y.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            logits = model(x)
            
            # Loss Calculation (Transpose to match CrossEntropy expectations)
            # logits: (batch, vocab, seq) | y: (batch, seq)
            loss = loss_fn(logits.transpose(1, 2), y)
            
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.25)
            
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            steps += 1
            
    # Validation Phase
    model.eval()
    model.reset() # Reset memory before validating!
    with torch.no_grad():
        vloss, v_steps = 0.0, 0
        for x, y in valid_ds:
            if x.shape == torch.Size([batch_size, seq_len]):
                x, y = x.to(device), y.to(device)
                logits = model(x)
                vloss += loss_fn(logits.transpose(1, 2), y).item()
                v_steps += 1
                
    model.reset() # Reset memory before next training epoch!
    
    train_loss_avg = total_loss / steps
    valid_loss_avg = vloss / v_steps
    current_lr = scheduler.get_last_lr()[0]
    
    print(f"epoch {e+1} | train: {train_loss_avg:.4f} | valid: {valid_loss_avg:.4f} | lr: {current_lr:.6f}")

epoch 1 | train: 8.1749 | valid: 7.4482 | lr: 0.000213
epoch 2 | train: 7.5658 | valid: 7.1558 | lr: 0.000253
epoch 3 | train: 7.2928 | valid: 6.9698 | lr: 0.000318
epoch 4 | train: 7.0932 | valid: 6.7854 | lr: 0.000408
epoch 5 | train: 6.9015 | valid: 6.5842 | lr: 0.000522
epoch 6 | train: 6.7074 | valid: 6.3848 | lr: 0.000660
epoch 7 | train: 6.5011 | valid: 6.1801 | lr: 0.000818
epoch 8 | train: 6.3018 | valid: 6.0224 | lr: 0.000996
epoch 9 | train: 6.1497 | valid: 5.9158 | lr: 0.001192
epoch 10 | train: 6.0126 | valid: 5.8483 | lr: 0.001403
epoch 11 | train: 5.8976 | valid: 5.7811 | lr: 0.001628
epoch 12 | train: 5.7994 | valid: 5.7621 | lr: 0.001863
epoch 13 | train: 5.7186 | valid: 5.7286 | lr: 0.002106
epoch 14 | train: 5.6339 | valid: 5.7165 | lr: 0.002354
epoch 15 | train: 5.5560 | valid: 5.7282 | lr: 0.002606
epoch 16 | train: 5.4952 | valid: 5.7387 | lr: 0.002857
epoch 17 | train: 5.4238 | valid: 5.7469 | lr: 0.003105
epoch 18 | train: 5.3708 | valid: 5.7701 | lr: 0.003348
e